# 03b · Irigat 2024: regula radar Sentinel-1 + pivoți geometrici (D26)
Fără clasificator optic și fără gardă pe NDVI de vară (evitarea circularității cu `loss_rel` din HR-VPP TPROD):

- **Pivoți geometrici (`data/out/pivots_2024.gpkg`):** $C_{\text{PP}} \ge 0{,}75$, $C_{\text{MBC}} \ge 0{,}60$, $A \ge 5\text{ ha}$, pe toate grupele.
- **Regula Sentinel-1 pe culturile de primăvară (`tip_cultura_2024 == 'primavara'`):** aceeași orbită relativă, $\Delta t = 12\text{ zile}$ (iunie–august 2024), ploaie ERA5-Land $< 2\text{ mm}$ în ultimele 5 zile și $< 5\text{ mm}$ pe toate cele 12 zile, eveniment de umezire la $\Delta\text{VV} \ge +1{,}5\text{ dB}$.
- **Clase finale pe primăvară (`data/out/irrigation_2024.parquet` și `data/out/irrigation_2024.gpkg`):**
  - `irigat`: $\ge 1$ eveniment S1 valid SAU pivot geometric;
  - `neirigat`: $0$ evenimente, cel puțin 3 tranziții uscate observate și $\max \Delta\text{VV} < 0{,}8\text{ dB}$;
  - `nedeterminat`: restul parcelelor de primăvară.

In [1]:
import os, sys, json
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import geometric_pivots, s1_spring_irrigation_rule
import numpy as np, pandas as pd, geopandas as gpd

P = gpd.read_parquet(OUT / 'parcels.parquet').to_crs(CRS)
T = pd.read_parquet(OUT / 'parcel_target.parquet')[['parcel_id', 'is_arable', 'tip_cultura_2024', 'loss_rel']]
S1 = pd.read_parquet(INTERIM / 'q2_s1_vv_arable.parquet')
ERA = pd.read_parquet(INTERIM / 'q2_era5_daily_arable.parquet')
print('Parcele:', len(P), '| S1 arabil:', S1.shape, '| ERA5 arabil:', ERA.shape)

Parcele: 17466 | S1 arabil: (13237, 35) | ERA5 arabil: (13237, 110)


In [2]:
# 1. Pivoți geometrici (strat separat, pe toate grupele, fără condiție NDVI)
piv_df = geometric_pivots(P.geometry)
P_piv = P[['parcel_id', 'geometry']].join(piv_df).merge(T, on='parcel_id')
P_piv['grup_afisare'] = np.where(P_piv.is_arable, P_piv.tip_cultura_2024, 'non-arabil')
piv_only = gpd.GeoDataFrame(P_piv[P_piv.is_pivot].copy(), geometry='geometry', crs=CRS)
piv_only.to_file(OUT / 'pivots_2024.gpkg', layer='pivoti', driver='GPKG')
piv_only.drop(columns='geometry').to_parquet(OUT / 'pivots_2024.parquet')
print(f'Total pivoți geometrici (C_PP>=0.75, C_MBC>=0.60, A>=5ha): {len(piv_only)} parcele ({piv_only.area_ha.sum():.1f} ha)')
for grp, gdf in piv_only.groupby('grup_afisare'):
    print(f'  {grp:14s}: {len(gdf):3d} parcele ({gdf.area_ha.sum():7.1f} ha)')

Total pivoți geometrici (C_PP>=0.75, C_MBC>=0.60, A>=5ha): 121 parcele (1810.1 ha)
  grup_incert   :   3 parcele (   45.1 ha)
  non-arabil    :  19 parcele (  119.0 ha)
  primavara     :  32 parcele (  369.8 ha)
  toamna        :  67 parcele ( 1276.2 ha)


In [3]:
# 2. Verificare tranziții S1 (12 zile, aceeași orbită) și clasificare pe culturile de primăvară 2024
D = P_piv.merge(S1, on='parcel_id', how='left').merge(ERA, on='parcel_id', how='left')
spring = D[D.is_arable & D.tip_cultura_2024.eq('primavara')].copy().reset_index(drop=True)

# Varianta A: fără excluderi automate (conform regulii „Nu exclude singur”)
tr_all, res_all = s1_spring_irrigation_rule(spring, exclude_transitions=())
tr_all.to_csv(OUT / 's1_transitions_2024.csv', index=False)
print('=== TRANZITII 12 ZILE CU PARCELE USCATE (rain_5d < 2 mm & rain_12d < 5 mm) ===')
cols_show = ['key', 'orbit', 't0', 't1', 'n_obs', 'n_dry', 'pct_dry_obs', 'rain5_med', 'rain12_med', 'dvv_med_dry', 'n_evt_dry', 'pct_evt_dry', 'pct_evt_obs']
print(tr_all[tr_all.n_dry > 0][cols_show].round(2).to_string(index=False))

# Varianta B: fără tranziția cu efect regional propusă spre excludere (o7_0615_0627)
_, res_no0627 = s1_spring_irrigation_rule(spring, exclude_transitions=('o7_0615_0627',))

for label, res in [('VARIANTA A (brut, fără excluderi automate)', res_all),
                   ('VARIANTA B (fără tranziția regională o7_0615_0627)', res_no0627)]:
    print(f'\n--- {label} ---')
    for c_name in ('irigat', 'neirigat', 'nedeterminat'):
        m = res.clasa_irigare.eq(c_name)
        n_p = int(m.sum())
        ha_p = float(spring.loc[m, 'area_ha'].sum())
        print(f'  {c_name:14s}: {n_p:5d} parcele ({100*n_p/len(spring):5.1f}%) | {ha_p:8.1f} ha ({100*ha_p/spring.area_ha.sum():5.1f}%)')

out_df = pd.DataFrame({
    'parcel_id': spring['parcel_id'],
    'area_ha': spring['area_ha'],
    'is_pivot': res_all['is_pivot'],
    'n_dry_obs': res_all['n_dry_obs'],
    'n_events_s1': res_all['n_events_s1'],
    'max_dvv_dry': res_all['max_dvv_dry'],
    'clasa_irigare': res_all['clasa_irigare'],
    'n_dry_obs_fara_0627': res_no0627['n_dry_obs'],
    'n_events_s1_fara_0627': res_no0627['n_events_s1'],
    'max_dvv_dry_fara_0627': res_no0627['max_dvv_dry'],
    'clasa_irigare_fara_0627': res_no0627['clasa_irigare'],
    'irrigated': res_all['clasa_irigare'].eq('irigat'),
})
out_df.to_parquet(OUT / 'irrigation_2024.parquet')
gpd.GeoDataFrame(out_df.merge(spring[['parcel_id', 'geometry']], on='parcel_id'), geometry='geometry', crs=CRS).to_file(
    OUT / 'irrigation_2024.gpkg', layer='irigare_primavara_2024', driver='GPKG'
)
metrics = {
    'method': 's1_vv_rule_plus_geometric_pivots_spring_only',
    'n_spring_parcels': int(len(spring)),
    'ha_spring_parcels': round(float(spring.area_ha.sum()), 2),
    'varianta_a_raw': {
        c: {'n': int(res_all.clasa_irigare.eq(c).sum()), 'ha': round(float(spring.loc[res_all.clasa_irigare.eq(c), 'area_ha'].sum()), 2)}
        for c in ('irigat', 'neirigat', 'nedeterminat')
    },
    'varianta_b_fara_o7_0615_0627': {
        c: {'n': int(res_no0627.clasa_irigare.eq(c).sum()), 'ha': round(float(spring.loc[res_no0627.clasa_irigare.eq(c), 'area_ha'].sum()), 2)}
        for c in ('irigat', 'neirigat', 'nedeterminat')
    },
}
(OUT / 'irrigation_metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')

=== TRANZITII 12 ZILE CU PARCELE USCATE (rain_5d < 2 mm & rain_12d < 5 mm) ===


          key  orbit         t0         t1  n_obs  n_dry  pct_dry_obs  rain5_med  rain12_med  dvv_med_dry  n_evt_dry  pct_evt_dry  pct_evt_obs
 o7_0615_0627      7 2024-06-15 2024-06-27   5836   4106        70.36       1.20        1.23         0.07        584        14.22        10.01
 o7_0802_0814      7 2024-08-02 2024-08-14   5836   5836       100.00       0.10        2.02        -0.19        190         3.26         3.26
 o7_0814_0826      7 2024-08-14 2024-08-26   5836   5836       100.00       0.04        1.89        -0.31        217         3.72         3.72
o29_0803_0815     29 2024-08-03 2024-08-15   5836   5836       100.00       0.08        1.34        -0.44        116         1.99         1.99
o29_0815_0827     29 2024-08-15 2024-08-27   5836   5836       100.00       0.27        2.08        -0.42        270         4.63         4.63
o80_0702_0714     80 2024-07-02 2024-07-14    179    179       100.00       0.25        1.45        -1.46          1         0.56         0.5

565